
# Ampliación del TFG de valoración de activos: revisión empírica del CAPM y el modelo de tres factores en datos anuales

### Trabajo Fin de Grado · Economía · Universidad de Santiago de Compostela

**Autor:** Miguel Nicolás Suárez Crespo
**Director:** Antonio Rodríguez Sampayo

---

Este Notebook constituye la ampliación online del TFG. Trata de la interpretación de los modelos CAPM y FF3 sobre las 100 carteras *Size × Book-to-Market* de Kenneth R. French, en frecuencia anual (1964–2025). El análisis en frecuencia mensual, así como sus respectivas interpretaciones, han sido expuestas en el cuerpo principal del TFG.

### Contenidos

1. Configuración y carga de datos
2. CAPM anual
3. FF3 anual (modelo de tres factores)
4. Comparativa mensual vs anual
5. Conclusiones

### Nota sobre la implementación

*Declaración: el código de este apéndice computacional ha sido generado con asistencia de Claude (Anthropic) y revisado, validado y redactado por el autor. Las decisiones metodológicas, las interpretaciones y las conclusiones son responsabilidad exclusiva del autor.*



## 1. Configuración

### 1.1 Bibliotecas

Importamos las bibliotecas necesarias para el análisis: `pandas` y `numpy` para manipulación de datos, `statsmodels` para las regresiones y los contrastes de heterocedasticidad, `scipy.stats` para los p-valores y `matplotlib` para los gráficos.


In [35]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.stats.diagnostic as smdiag
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:.4f}'.format)
np.set_printoptions(precision=4, suppress=True)

print("Bibliotecas cargadas correctamente.")


Bibliotecas cargadas correctamente.


In [44]:
# Rutas y constantes globales
CARPETA_DATOS = "datos"
VENTANA_A = 10

print(f"Carpeta de datos: {CARPETA_DATOS}")
print(f"Ventana movil: {VENTANA_A} anos")

Carpeta de datos: datos
Ventana movil: 10 anos



### 1.2 Carga de los datos brutos

Los datos provienen de la base de datos pública de Kenneth R. French. Empleamos los dos archivos que se emplearon en el texto principal del TFG:

- `F-F_Research_Data_Factors.CSV`: los tres factores de Fama y French (Mkt-RF, SMB, HML) y el tipo libre de riesgo (RF), tanto en frecuencia mensual como anual.
- `100_Portfolios_10x10.CSV`: rendimientos *value-weighted* de las 100 carteras Size×BE/ME, en frecuencia mensual y anual.



## 2. CAPM anual

Replicamos el análisis del CAPM desarrollado en el texto principal del TFG con datos anuales. El periodo muestral pasa a ser 1964–2025 ($T = 62$ años), y la ventana móvil se ajusta a 10 años

### 2.1 Carga de datos anuales


In [48]:
# Factores anuales
factores_a = pd.read_csv(os.path.join(CARPETA_DATOS, "datos_factores.csv"),
                          skiprows=1205, index_col=0)
factores_a.index = factores_a.index.astype(str).str.strip()
factores_a = factores_a[factores_a.index.str.match(r"^\d{4}$")]
factores_a.index = factores_a.index.astype(int)
factores_a = factores_a.apply(pd.to_numeric, errors="coerce") / 100

# Carteras anuales VW
carteras_a = pd.read_csv(os.path.join(CARPETA_DATOS, "datos_100_carteras.csv"),
                          skiprows=2419, nrows=100, index_col=0)
carteras_a.index = carteras_a.index.astype(str).str.strip()
carteras_a = carteras_a[carteras_a.index.str.match(r"^\d{4}$")]
carteras_a.index = carteras_a.index.astype(int)
carteras_a = carteras_a.apply(pd.to_numeric, errors="coerce")
carteras_a = carteras_a.replace([-99.99, -999, -100], np.nan) / 100

INICIO_MUESTRAL_A  = 1964
INICIO_EXTENDIDO_A = 1954

factores_a_ext = factores_a.loc[INICIO_EXTENDIDO_A:].copy()
carteras_a_ext = carteras_a.loc[INICIO_EXTENDIDO_A:].copy()
anios_ext = factores_a_ext.index.intersection(carteras_a_ext.index)
factores_a_ext = factores_a_ext.loc[anios_ext]
carteras_a_ext = carteras_a_ext.loc[anios_ext]

factores_a = factores_a.loc[INICIO_MUESTRAL_A:]
carteras_a = carteras_a.loc[INICIO_MUESTRAL_A:]
anios_a = factores_a.index.intersection(carteras_a.index)
factores_a = factores_a.loc[anios_a]
carteras_a = carteras_a.loc[anios_a]

rf_a       = factores_a["RF"]
mkt_rf_a   = factores_a["Mkt-RF"]
ff3_a      = factores_a[["Mkt-RF", "SMB", "HML"]]
excesos_a  = carteras_a.sub(rf_a, axis=0)

rf_a_ext      = factores_a_ext["RF"]
mkt_rf_a_ext  = factores_a_ext["Mkt-RF"]
ff3_a_ext     = factores_a_ext[["Mkt-RF", "SMB", "HML"]]
excesos_a_ext = carteras_a_ext.sub(rf_a_ext, axis=0)

print(f"Datos ANUALES cargados:")
print(f"  Periodo muestral:                      {anios_a.min()} a {anios_a.max()}  (T = {len(anios_a)})")
print(f"  Periodo extendido para betas móviles:  {anios_ext.min()} a {anios_ext.max()}")
print(f"  Carteras:                              N = {excesos_a.shape[1]}")


Datos ANUALES cargados:
  Periodo muestral:                      1964 a 2025  (T = 62)
  Periodo extendido para betas móviles:  1954 a 2025
  Carteras:                              N = 100


### 2.2 CAPM anual: implementación de las dos etapas

A continuación se replica la metodología en dos etapas de Fama y MacBeth (1973) sobre los datos anuales, siguiendo el mismo procedimiento de la sección 4 del TFG para los datos mensuales. Las celdas siguientes ejecutan, por orden:

1. La **primera etapa estática**: una regresión MCO por cartera sobre los 62 años del periodo muestral, de la que se obtienen los descriptivos (betas, alfas y R²).

2. La **primera etapa con ventana móvil de 10 años**, que reestima las betas año a año sobre los 10 años anteriores. Constituye la entrada para la segunda etapa.

3. La **segunda etapa de Fama-MacBeth** sobre las betas móviles, de la que se obtienen los coeficientes $\hat{\gamma}_0$ y $\hat{\gamma}_1$ con su correspondiente inferencia.

Una particularidad del análisis anual: el test GRS no es aplicable porque con $T = 62 < N = 100$, la matriz de covarianzas residuales $\Sigma$ no se puede invertir. Es una limitación estructural del análisis anual con 100 carteras, no un problema metodológico, y el rechazo formal del CAPM queda apoyado por el análisis mensual.

#### 2.2.1 Primera etapa estática CAPM con datos anuales

In [49]:
# Primera etapa estática CAPM anual
resultados_capm_a = []
for cartera in excesos_a.columns:
    y = excesos_a[cartera]
    datos = pd.concat([y, mkt_rf_a.rename("Mkt-RF")], axis=1).dropna()
    X = sm.add_constant(datos["Mkt-RF"])
    m = sm.OLS(datos[cartera], X).fit()
    resultados_capm_a.append({
        "cartera":  cartera,
        "alpha":    m.params["const"],
        "beta":     m.params["Mkt-RF"],
        "alpha_t":  m.tvalues["const"],
        "r2":       m.rsquared,
        "sigma_resid": np.sqrt(m.mse_resid),
        "rend_medio":  datos[cartera].mean(),
    })
primera_capm_a = pd.DataFrame(resultados_capm_a).set_index("cartera")

print("=" * 70)
print("PRIMERA ETAPA ESTÁTICA - CAPM ANUAL (T = 62)")
print("=" * 70)
print(f"\nBeta:    media = {primera_capm_a['beta'].mean():.3f}   "
      f"min = {primera_capm_a['beta'].min():.3f}   "
      f"max = {primera_capm_a['beta'].max():.3f}")
print(f"R²:      media = {primera_capm_a['r2'].mean():.3f}")
print(f"\nAlfas significativos al 5%:  {(primera_capm_a['alpha_t'].abs() > 1.96).sum()}/100")
print(f"Alfas significativos al 1%:  {(primera_capm_a['alpha_t'].abs() > 2.58).sum()}/100")


PRIMERA ETAPA ESTÁTICA - CAPM ANUAL (T = 62)

Beta:    media = 1.012   min = 0.701   max = 1.590
R²:      media = 0.561

Alfas significativos al 5%:  22/100
Alfas significativos al 1%:  11/100


El R² medio (0,561) es algo inferior al mensual (0,645). Los 22 alfas significativos al 5% y 11 al 1% superan ampliamente lo esperado bajo la hipótesis nula (5 y 1, respectivamente), anticipando el rechazo del CAPM ya en frecuencia anual.


#### 2.2.2 Diagnóstico de heterocedasticidad (anual)

Aplicamos el test de White también sobre los residuos de las 100 regresiones de la primera etapa estática anual.


In [52]:
# Test de White CAPM anual
X_t = sm.add_constant(mkt_rf_a)
test_white_a = []
for cartera in excesos_a.columns:
    y = excesos_a[cartera]
    datos = pd.concat([y, X_t], axis=1).dropna()
    Xc = datos[["const", "Mkt-RF"]]
    m = sm.OLS(datos[cartera], Xc).fit()
    try:
        w_lm, w_lm_p, _, _ = smdiag.het_white(m.resid, Xc)
    except Exception:
        w_lm, w_lm_p = np.nan, np.nan
    test_white_a.append({"cartera": cartera, "W": w_lm, "p": w_lm_p})

white_a = pd.DataFrame(test_white_a).set_index("cartera")
n_5_a = (white_a["p"] < 0.05).sum()
n_1_a = (white_a["p"] < 0.01).sum()

print("=" * 70)
print("CONTRASTE DE WHITE (CAPM ANUAL)")
print("=" * 70)
print(f"\nRechazan H0 al 5%:  {n_5_a}/100   (mensual: 82/100)")
print(f"Rechazan H0 al 1%:  {n_1_a}/100   (mensual: 74/100)")



CONTRASTE DE WHITE (CAPM ANUAL)

Rechazan H0 al 5%:  4/100   (mensual: 82/100)
Rechazan H0 al 1%:  0/100   (mensual: 74/100)


Con datos anuales se detecta mucha menos heterocedasticidad, como consecuencia de la agregación temporal: las varianzas condicionales mensuales se promedian en anuales y la dependencia temporal queda atenuada.


#### 2.2.3 Primera etapa y segunda etapa con ventana móvil de 10 años

Replicamos la metodología móvil con ventana de 10 años, para su comparación con la versión mensual. La primera ventana (1964) emplea los datos desde 1954. 


In [53]:
# Primera etapa rolling 10 anos
betas_a_rolling = pd.DataFrame(index=excesos_a.index, columns=excesos_a.columns, dtype=float)
sigma_resid_a_rolling = pd.DataFrame(index=excesos_a.index, columns=excesos_a.columns, dtype=float)
for anio_t in excesos_a.index:
    pos_ext = excesos_a_ext.index.get_loc(anio_t)
    anios_ventana = excesos_a_ext.index[pos_ext - VENTANA_A : pos_ext]
    mkt_ventana = mkt_rf_a_ext.loc[anios_ventana]
    X_ventana = sm.add_constant(mkt_ventana.rename("Mkt-RF"))
    for cartera in excesos_a_ext.columns:
        y_ventana = excesos_a_ext.loc[anios_ventana, cartera]
        datos = pd.concat([y_ventana, X_ventana], axis=1).dropna()
        if len(datos) < 6:
            continue
        m = sm.OLS(datos.iloc[:, 0], datos.iloc[:, 1:]).fit()
        betas_a_rolling.loc[anio_t, cartera] = m.params["Mkt-RF"]
        sigma_resid_a_rolling.loc[anio_t, cartera] = np.sqrt(m.mse_resid)

print("=" * 70)
print(f"BETAS CON VENTANA MOVIL CAPM ANUAL (ventana = {VENTANA_A} anos)")
print("=" * 70)
print(f"\nBetas rolling estimadas: {betas_a_rolling.shape[0]} anos x {betas_a_rolling.shape[1]} carteras")
print(f"({betas_a_rolling.shape[0] * betas_a_rolling.shape[1]:,} regresiones de serie temporal)")
print(f"\nDispersion transversal de las betas:")
print(f"  Media:   {betas_a_rolling.stack().mean():.3f}")
print(f"  Minima:  {betas_a_rolling.stack().min():.3f}")
print(f"  Maxima:  {betas_a_rolling.stack().max():.3f}")

# Segunda etapa movil
gammas_capm_a = []
anios_t = []
for anio_t in betas_a_rolling.index:
    y = excesos_a.loc[anio_t]
    beta_t = betas_a_rolling.loc[anio_t]
    X = sm.add_constant(beta_t.rename("beta"))
    datos = pd.concat([y, X], axis=1).dropna()
    if len(datos) < 10:
        continue
    m = sm.OLS(datos.iloc[:, 0], datos.iloc[:, 1:]).fit()
    gammas_capm_a.append((m.params["const"], m.params["beta"]))
    anios_t.append(anio_t)

gammas_capm_a = pd.DataFrame(gammas_capm_a, columns=["gamma_0", "gamma_1"], index=anios_t)
T_a = len(gammas_capm_a)

medias_a = gammas_capm_a.mean()
se_a = gammas_capm_a.std() / np.sqrt(T_a)
t_a = medias_a / se_a
p_a = pd.Series(2 * (1 - stats.t.cdf(t_a.abs(), df=T_a-1)), index=t_a.index)

print("\n" + "=" * 70)
print(f"SEGUNDA ETAPA - CAPM ANUAL (T = {T_a})")
print("=" * 70)
print(f"  gamma_0 = {medias_a['gamma_0']*100:.3f}% anual  (t = {t_a['gamma_0']:.2f}, p = {p_a['gamma_0']:.4f})")
print(f"  gamma_1 = {medias_a['gamma_1']*100:.3f}% anual  (t = {t_a['gamma_1']:.2f}, p = {p_a['gamma_1']:.4f})")
print(f"  Prima realizada del mercado: {mkt_rf_a.mean()*100:.3f}% anual")

BETAS CON VENTANA MOVIL CAPM ANUAL (ventana = 10 anos)

Betas rolling estimadas: 62 anos x 100 carteras
(6,200 regresiones de serie temporal)

Dispersion transversal de las betas:
  Media:   1.098
  Minima:  -0.492
  Maxima:  3.327

SEGUNDA ETAPA - CAPM ANUAL (T = 62)
  gamma_0 = 9.846% anual  (t = 3.95, p = 0.0002)
  gamma_1 = -0.633% anual  (t = -0.37, p = 0.7145)
  Prima realizada del mercado: 7.694% anual


El intercepto $\hat{\gamma}_0 \approx 9{,}85\%$ anual resulta significativamente distinto de cero, mientras que la pendiente $\hat{\gamma}_1 \approx -0{,}63\%$ es estadísticamente indistinguible de cero, alejándose de la prima realizada del mercado ($7{,}69\%$). Estos resultados reproducen el patrón del análisis mensual: rechazo de la hipótesis $\gamma_0 = 0$ y ausencia de una prima de mercado detectable en la sección cruzada, manifestación de la anomalía de la LMA plana, documentada por Black, Jensen y Scholes (1972).


## 3. FF3 anual

Replicamos el FF3 con datos anuales, manteniendo una ventana móvil de 10 años.

### 3.1 Primera etapa estática multifactorial


In [57]:
# Primera etapa estática FF3 anual
resultados_ff3_a = []
for cartera in excesos_a.columns:
    y = excesos_a[cartera]
    datos = pd.concat([y, ff3_a], axis=1).dropna()
    X = sm.add_constant(datos[["Mkt-RF", "SMB", "HML"]])
    m = sm.OLS(datos.iloc[:, 0], X).fit()
    resultados_ff3_a.append({
        "cartera":  cartera,
        "alpha":    m.params["const"],
        "beta_MKT": m.params["Mkt-RF"],
        "beta_SMB": m.params["SMB"],
        "beta_HML": m.params["HML"],
        "alpha_t":  m.tvalues["const"],
        "r2":       m.rsquared,
    })
primera_ff3_a = pd.DataFrame(resultados_ff3_a).set_index("cartera")

print("=" * 70)
print("PRIMERA ETAPA ESTÁTICA - FF3 ANUAL (T = 62)")
print("=" * 70)
print()
for k in ["beta_MKT", "beta_SMB", "beta_HML"]:
    print(f"{k}:   media = {primera_ff3_a[k].mean():.3f},   "
          f"min = {primera_ff3_a[k].min():.3f},   "
          f"max = {primera_ff3_a[k].max():.3f}")
print(f"\nR² medio:   {primera_ff3_a['r2'].mean():.3f}   "
      f"(frente a 0.798 en mensual)")
print(f"\nAlfas significativos al 5%:  {(primera_ff3_a['alpha_t'].abs() > 1.96).sum()}/100")
print(f"Alfas significativos al 1%:  {(primera_ff3_a['alpha_t'].abs() > 2.58).sum()}/100")


PRIMERA ETAPA ESTÁTICA - FF3 ANUAL (T = 62)

beta_MKT:   media = 0.980,   min = 0.717,   max = 1.285
beta_SMB:   media = 0.599,   min = -0.478,   max = 1.912
beta_HML:   media = 0.284,   min = -0.717,   max = 1.016

R² medio:   0.798   (frente a 0.798 en mensual)

Alfas significativos al 5%:  15/100
Alfas significativos al 1%:  5/100


Las tres betas factoriales presentan medias transversales positivas sobre las 100 carteras:

* $\beta_{\text{MKT}}$: Su media de $0{.}99$ indica que, en promedio, las carteras se mueven al mismo ritmo que el mercado.
* $\beta_{\text{SMB}}$: Con una media de $0{.}55$, presenta valores que van desde $-1{.}20$ (carteras de empresas grandes) hasta $3{,}11$ (carteras de empresas pequeñas), reflejando la dispersión natural en la matriz $\text{Size} \times \text{BM}$.
* $\beta_{\text{HML}}$: Su media de $0{.}15$ sigue la misma lógica respecto al factor valor: las carteras de empresas *value* tienen una $\beta_{\text{HML}}$ positiva, mientras que las de tipo *growth* la tienen negativa.

El R² medio es prácticamente idéntico al mensual y mayor que el del CAPM anual (0,561), lo que evidencia la mayor capacidad explicativa del modelo multifactorial. 
Los alfas significativos (15/100 al 5% y 5/100 al 1%) también se reducen respecto al CAPM anual (22 y 11), pero siguen por encima de lo esperado bajo la hipótesis nula.


### 3.2 Diagnóstico de heterocedasticidad FF3 (anual)


In [58]:
# Test de White FF3 anual
X_t = sm.add_constant(ff3_a)
test_white_ff3_a = []
for cartera in excesos_a.columns:
    y = excesos_a[cartera]
    datos = pd.concat([y, X_t], axis=1).dropna()
    Xc = datos[["const", "Mkt-RF", "SMB", "HML"]]
    m = sm.OLS(datos[cartera], Xc).fit()
    try:
        w_lm, w_lm_p, _, _ = smdiag.het_white(m.resid, Xc)
    except Exception:
        w_lm, w_lm_p = np.nan, np.nan
    test_white_ff3_a.append({"cartera": cartera, "W": w_lm, "p": w_lm_p})

white_ff3_a = pd.DataFrame(test_white_ff3_a).set_index("cartera")
n_5_ff3_a = (white_ff3_a["p"] < 0.05).sum()
n_1_ff3_a = (white_ff3_a["p"] < 0.01).sum()

print("=" * 70)
print("CONTRASTE DE WHITE (FF3 ANUAL)")
print("=" * 70)
print(f"\nRechazan H0 al 5%:  {n_5_ff3_a}/100   (mensual FF3: 95/100)")
print(f"Rechazan H0 al 1%:  {n_1_ff3_a}/100   (mensual FF3: 90/100)")


CONTRASTE DE WHITE (FF3 ANUAL)

Rechazan H0 al 5%:  37/100   (mensual FF3: 95/100)
Rechazan H0 al 1%:  20/100   (mensual FF3: 90/100)


Igual que en CAPM anual, la agregación temporal reduce la evidencia de heterocedasticidad. Sin embargo, en el modelo de tres factores el resultado es peor. La metodología con ventana móvil sigue siendo conceptualmente preferible.


### 3.3 Primera y segunda etapa con ventana móvil.


In [59]:
# Primera etapa rolling 10 años FF3
beta_MKT_a_r = pd.DataFrame(index=excesos_a.index, columns=excesos_a.columns, dtype=float)
beta_SMB_a_r = pd.DataFrame(index=excesos_a.index, columns=excesos_a.columns, dtype=float)
beta_HML_a_r = pd.DataFrame(index=excesos_a.index, columns=excesos_a.columns, dtype=float)
for anio_t in excesos_a.index:
    pos_ext = excesos_a_ext.index.get_loc(anio_t)
    anios_ventana = excesos_a_ext.index[pos_ext - VENTANA_A : pos_ext]
    factores_ventana = ff3_a_ext.loc[anios_ventana]
    X_ventana = sm.add_constant(factores_ventana)
    for cartera in excesos_a_ext.columns:
        y_ventana = excesos_a_ext.loc[anios_ventana, cartera]
        datos = pd.concat([y_ventana, X_ventana], axis=1).dropna()
        if len(datos) < 6:
            continue
        m = sm.OLS(datos.iloc[:, 0], datos.iloc[:, 1:]).fit()
        beta_MKT_a_r.loc[anio_t, cartera] = m.params["Mkt-RF"]
        beta_SMB_a_r.loc[anio_t, cartera] = m.params["SMB"]
        beta_HML_a_r.loc[anio_t, cartera] = m.params["HML"]

print(f"Betas rolling FF3 anuales estimadas:")
print(f"  beta_MKT: media={beta_MKT_a_r.stack().mean():.3f}   min={beta_MKT_a_r.stack().min():.3f}   max={beta_MKT_a_r.stack().max():.3f}")
print(f"  beta_SMB: media={beta_SMB_a_r.stack().mean():.3f}   min={beta_SMB_a_r.stack().min():.3f}   max={beta_SMB_a_r.stack().max():.3f}")
print(f"  beta_HML: media={beta_HML_a_r.stack().mean():.3f}   min={beta_HML_a_r.stack().min():.3f}   max={beta_HML_a_r.stack().max():.3f}")

# Segunda etapa móvil FF3
gammas_ff3_a_list = []
anios_t = []
for anio_t in beta_MKT_a_r.index:
    y = excesos_a.loc[anio_t]
    X = pd.concat([
        pd.Series(1.0, index=excesos_a.columns, name="const"),
        beta_MKT_a_r.loc[anio_t].rename("beta_MKT"),
        beta_SMB_a_r.loc[anio_t].rename("beta_SMB"),
        beta_HML_a_r.loc[anio_t].rename("beta_HML"),
    ], axis=1)
    datos = pd.concat([y, X], axis=1).dropna()
    if len(datos) < 10:
        continue
    m = sm.OLS(datos.iloc[:, 0], datos.iloc[:, 1:]).fit()
    gammas_ff3_a_list.append(m.params.values)
    anios_t.append(anio_t)

gammas_ff3_a = pd.DataFrame(gammas_ff3_a_list,
                             columns=["gamma_0", "gamma_MKT", "gamma_SMB", "gamma_HML"],
                             index=anios_t)
T_a_ff3 = len(gammas_ff3_a)
medias_ff3_a = gammas_ff3_a.mean()
se_ff3_a = gammas_ff3_a.std() / np.sqrt(T_a_ff3)
t_ff3_a = medias_ff3_a / se_ff3_a
p_ff3_a = pd.Series(2 * (1 - stats.t.cdf(t_ff3_a.abs(), df=T_a_ff3-1)), index=t_ff3_a.index)

print("\n" + "=" * 70)
print(f"SEGUNDA ETAPA - FF3 ANUAL (T = {T_a_ff3})")
print("=" * 70)
print(f"\n{'':12} {'media anual (%)':>15} {'t-stat':>8} {'p-valor':>8}")
for k in ["gamma_0", "gamma_MKT", "gamma_SMB", "gamma_HML"]:
    print(f"{k:<12} {medias_ff3_a[k]*100:>15.3f}  {t_ff3_a[k]:>8.2f} {p_ff3_a[k]:>8.4f}")


Betas rolling FF3 anuales estimadas:
  beta_MKT: media=1.002   min=-0.027   max=2.327
  beta_SMB: media=0.576   min=-2.482   max=4.210
  beta_HML: media=0.247   min=-2.619   max=3.956

SEGUNDA ETAPA - FF3 ANUAL (T = 62)

             media anual (%)   t-stat  p-valor
gamma_0                8.306      3.76   0.0004
gamma_MKT             -0.161     -0.14   0.8919
gamma_SMB              1.318      1.17   0.2460
gamma_HML              3.092      2.23   0.0296


En la primera etapa móvil, las betas se reestiman cada año usando los 10 años anteriores.

Para cada año $t$ se estima la regresión transversal:

$$r_{j,t} - r_{f,t} = \gamma_{0,t} + \gamma_{MKT,t}\,\hat{\beta}_{j,MKT,t} + \gamma_{SMB,t}\,\hat{\beta}_{j,SMB,t} + \gamma_{HML,t}\,\hat{\beta}_{j,HML,t} + \eta_{j,t}$$

El intercepto $\hat{\gamma}_0$ es significativamente distinto de cero, rechazando una de las predicciones del FF3. La prima del factor de mercado $\hat{\gamma}_{MKT}$ es estadísticamente indistinguible de cero, reproduciendo la anomalía de la LMA plana ya documentada en el CAPM anual. La prima del factor tamaño $\hat{\gamma}_{SMB}$ no resulta significativa, aunque mantiene el signo positivo esperado. Por otro lado, la prima del factor valor $\hat{\gamma}_{HML}$ es positiva y significativa al 5%, próxima a la prima realizada del factor HML (3,66% anual): el efecto valor sí tiene precio en la sección cruzada anual.


## 4. Comparativa global mensual vs anual

### CAPM: comparativa mensual vs anual

| | $\hat{\gamma}_0$ anual (%) | $\hat{\gamma}_1$ anual (%) | $t(\hat{\gamma}_0)$ | $t(\hat{\gamma}_1)$ |
|---|---|---|---|---|
| **CAPM mensual** ($T=754$) | 9,56 *** | −0,71 | 4,07 | −0,26 |
| **CAPM anual** ($T=62$) | 9,85 *** | −0,63 | 3,95 | −0,37 |
| Predicción CAPM | 0 | +7,17 (mensual) / +7,69 (aanual) | — | — |

### FF3: comparativa mensual vs anual

| | $\hat{\gamma}_0$ | $\hat{\gamma}_{MKT}$ | $\hat{\gamma}_{SMB}$ | $\hat{\gamma}_{HML}$ |
|---|---|---|---|---|
| **FF3 mensual** | 10,62 *** | −3,33 (*) | 1,55 | 2,91 ** |
| **FF3 anual** | 8,31 *** | −0,16 | 1,32 | 3,09 ** |

1. Las conclusiones se mantienen entre frecuencias: $\gamma_0$ significativamente positivo, $\gamma_1$ del CAPM indistinguible de cero, $\gamma_{HML}$ positivo y significativo.

2. El test GRS no es aplicable con datos anuales ($T = 62 < N = 100$), lo que es una limitación estructural del análisis anual con 100 carteras. La validez del rechazo conjunto del CAPM y del FF3 queda por el análisis mensual.

3. El test de White detecta menos heterocedasticidad en datos anuales (CAPM: 4/100 frente a 82/100; FF3: 37/100 frente a 95/100 al 5%), reflejo de que la agregación temporal suaviza la dependencia de la varianza condicional.




## 5. Conclusión final del estudio empírico del CAPM y FF3 

Este Notebook ha replicado y ampliado el contraste empírico del CAPM y del modelo de tres factores de Fama y French del TFG, incorporando un análisis con datos anuales (1964–2025) que sigue la misma metodología que el análisis principal mensual.

1. El CAPM se rechaza en el análisis mensual por dos vías formales independientes (test GRS y contraste de linealidad) y por la ausencia de una prima de mercado detectable en la sección cruzada, manifestación de la anomalía de la LMA plana (Black, Jensen y Scholes, 1972). El patrón sistemático de las desviaciones a lo largo del cociente BE/ME revela una limitación estructural del modelo de un solo factor.

2. El modelo de tres factores mejora sustancialmente la capacidad explicativa ($R^2$ medio del 0.80 frente a 0.65 del CAPM) y valida empíricamente el factor valor ($\hat{\gamma}_{HML}$ positivo y significativo). Sin embargo, persisten desviaciones sistemáticas: el GRS sigue rechazando y la anomalía de la LMA plana se mantiene, lo que sugiere la existencia de fuentes de riesgo no recogidas en el modelo.

3. El análisis anual confirma las conclusiones del análisis mensual: $\hat{\gamma}_0$ significativamente positivo, $\hat{\gamma}_1$ indistinguible de cero, $\hat{\gamma}_{HML}$ positivo y significativo. La única limitación específica del análisis anual es la no aplicabilidad del test GRS ($T = 62 < N = 100$). Esta coherencia entre frecuencias refuerza la robustez del análisis principal del TFG.

**Líneas naturales de investigación futura:**

- Contraste empírico del modelo de cinco factores de Fama y French (2015), que añade la rentabilidad operativa y el patrón de inversión empresarial como factores adicionales.

---

*Fin del Notebook.*